In [1]:
import numpy as np
import matplotlib.pyplot as plt
#%matplotlib qt
import pandas as pd
import seaborn as sns
from scipy.stats import norm
from pathlib import Path
import os
from scipy.io import wavfile
from pathlib import Path

In [2]:
# SUBHARMONICS DURATION : SIGNAL DURATION RATIO
def subharmRatio(subharm_total_no : int, subharms, signal_duration):
    subharm_ratio = 0.
    subharm_per_second = 0.
    subharm_durations = []

    density_slope = 0.
    pct_subharm_num_second_half = 0.
    dur_density_slope = 0.
    pct_subharm_dur_second_half = 0.

    if subharm_total_no > 0:
        subharm_durations = [b - a for a, b in subharms]

        total_subharm_dur = sum(subharm_durations)
        subharm_ratio = total_subharm_dur/signal_duration
        subharm_per_second = subharm_total_no / signal_duration

        # Second half features
        half_time = signal_duration / 2.0
        n_second = sum(1 for (a, b) in subharms if a >= half_time)
        n_first = subharm_total_no - n_second

        pct_subharm_num_second_half = n_second/subharm_total_no
        density_slope = (n_second - n_first)/half_time

        subharm_dur_second = 0.0
        for a, b in subharms:
            subharm_dur_second += max(0.0, b - max(a, half_time))
        subharm_dur_first = total_subharm_dur - subharm_dur_second

        pct_subharm_dur_second_half = subharm_dur_second / total_subharm_dur
        dur_density_slope = (subharm_dur_second - subharm_dur_first)/half_time

    if subharm_ratio >= 1.0:
        raise ValueError("Subharmonic-signal ratio larger than 1.")

    return subharm_ratio, subharm_per_second, subharm_durations, density_slope, pct_subharm_num_second_half, dur_density_slope, pct_subharm_dur_second_half

# FIRST SUBHARMONIC OCCURENCE
def firstSubharmOccur(subharm_total_no, subharms, signal_duration):
    first_subharm_occur = signal_duration
    if subharm_total_no > 0:
        first_subharm_occur = subharms[0][0]
    
    return first_subharm_occur

# MEAN / MEDIAN / STD / CV SUBHARMONIC DURATION
def statsSubharmDur(subharm_total_no, subharm_durations):
    avg_subharm_dur, median_subharm_dur, std_subharm_dur, coeffOfVar_subharm_dur, longest_subharm_dur = 0., 0., np.nan, np.nan, 0.

    if subharm_total_no > 0:
        avg_subharm_dur = np.mean(subharm_durations)
        median_subharm_dur = np.median(subharm_durations)

        if subharm_total_no > 1: # at least 2 subharmonics for std calculation
            std_subharm_dur = np.std(subharm_durations, ddof=1)
            coeffOfVar_subharm_dur = std_subharm_dur / avg_subharm_dur

        longest_subharm_dur = np.max(subharm_durations)

    return avg_subharm_dur, median_subharm_dur, std_subharm_dur, coeffOfVar_subharm_dur, longest_subharm_dur


# INTER-SUBHARMONIC INTERVALS 
def interSubharmIntervals(subharm_total_no, subharms):

    mean_intervals, median_intervals, std_intervals, COV_intervals = np.nan, np.nan, np.nan, np.nan

    if subharm_total_no > 1: # at least 2 subharmonics for an interval
        inter_intervals = subharms[1:, 0] - subharms[:-1, 1]

        mean_intervals = np.mean(inter_intervals)
        median_intervals = np.median(inter_intervals)

        if subharm_total_no > 2: # at least 2 intervals for std calculation
            std_intervals = np.std(inter_intervals, ddof=1)
            COV_intervals = std_intervals / mean_intervals

    return mean_intervals, median_intervals, std_intervals, COV_intervals

In [3]:
def interpolate_pitch(pitch):
    pitch = pitch.copy()
    pitch = np.array(pitch)
    pitch[pitch <= 0] = np.nan
    valid = np.isfinite(pitch) & (pitch > 0)

    if np.sum(valid) < 2:
        return pitch

    pitch_interp = np.interp(
        np.arange(len(pitch)),
        np.where(valid)[0],
        pitch[valid]
    )

    return pitch_interp

def getFrameXt(pitch, frame_idx, n_side, fref_mode): # maybe test win_len: 100 ms, 150 ms, 200 ms, 300 ms

    win_start = frame_idx-n_side
    win_end = frame_idx+n_side
    # print(f"start: {win_start}, end: {win_end}")
    window_pitch = pitch[win_start:win_end+1]

    local_frame_idx = n_side
    if np.isnan(window_pitch[local_frame_idx]):
        return None
    
    window_pitch_valid = window_pitch[(~np.isnan(window_pitch)) & (window_pitch > 0)]
    # if len(window_pitch_valid) < 3:
    #     return None

    # print(window_pitch)
    # print(window_pitch_valid)
    
    if fref_mode == "upper_60_80_median":
        p60 = np.percentile(window_pitch_valid, 60)
        p80 = np.percentile(window_pitch_valid, 80)
        fref = np.median(window_pitch_valid[(window_pitch_valid >= p60) & (window_pitch_valid <= p80)])
    elif fref_mode == "upper_70_90_median":
        p70 = np.percentile(window_pitch_valid, 70)
        p90 = np.percentile(window_pitch_valid, 90)
        fref = np.median(window_pitch_valid[(window_pitch_valid >= p70) & (window_pitch_valid <= p90)])
    elif fref_mode == "upper_80_90_median":
        p80 = np.percentile(window_pitch_valid, 80)
        p90 = np.percentile(window_pitch_valid, 90)
        fref = np.median(window_pitch_valid[(window_pitch_valid >= p80) & (window_pitch_valid <= p90)])
    elif fref_mode == "median": # median
        fref = np.median(window_pitch_valid)
    else:
        raise ValueError("Unsupported fref_mode")
    
    # print(f"f_ref = {fref}")
    if not np.isfinite(fref) or fref <= 0:
        return None
    x_frame = np.log2(window_pitch[local_frame_idx]/fref)
    # print(np.log2(window_pitch/fref))
    # print(x_frame)
    return x_frame


def getXtFrames(pitch, first_frame_idx, last_frame_idx, n_side, fref_mode):
    x_frames = []
    for frame_idx in range(first_frame_idx, last_frame_idx+1):
        x_t = getFrameXt(pitch, frame_idx, n_side, fref_mode)
        if x_t is None:
            print("NONE")
        x_frames.append(x_t)
    
    return np.array(x_frames)

In [4]:
def gaussian_pdf(x, mu, sigma):
    sigma = max(float(sigma), 1e-6)
    return (1.0 / (np.sqrt(2.0 * np.pi) * sigma)) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def em_two_fixed_means(x_frames, prior_non_init, prior_sub_init, max_iter=200, tol=1e-4, sigma_floor=0.05,
                       mu_non=0.0, mu_sub=-1.0, sigma_non_init=0.15, sigma_sub_init=0.15,
                       ):
    
    x = np.asarray(x_frames, dtype=float)
    x = x[np.isfinite(x)]

    if x.size < 5:
        raise ValueError("Not enough valid x_t samples for EM.")

    prior_non = float(prior_non_init)
    prior_sub = float(prior_sub_init)
    sigma_non = max(float(sigma_non_init), sigma_floor)
    sigma_sub = max(float(sigma_sub_init), sigma_floor)

    prev_posterior_sub = None

    for i in range(max_iter):
        # print(f"ITERATION {i+1}")

        # E-step
        p_non = prior_non * gaussian_pdf(x, mu_non, sigma_non)
        p_sub = prior_sub * gaussian_pdf(x, mu_sub, sigma_sub)
      
        posterior_non = p_non / (p_non + p_sub + 1e-12)
        posterior_sub = p_sub / (p_non + p_sub + 1e-12)

        # convergence check right after E-step
        if prev_posterior_sub is not None:
            max_posterior_change = np.max(np.abs(posterior_sub - prev_posterior_sub))
            # print(f"max posterior change = {max_posterior_change:.5f}")

            if max_posterior_change < tol:
                # print(f"Stopping: max posterior change = {max_posterior_change:.5f}")
                break
        
        prev_posterior_sub = posterior_sub.copy()

        # M-step
        N_non = np.sum(posterior_non)
        N_sub = np.sum(posterior_sub)
        
        prior_non = N_non / x.size
        prior_sub = N_sub / x.size

        sigma_non = np.sqrt(np.sum(posterior_non * (x - mu_non) ** 2) / (N_non + 1e-12))
        sigma_non = max(float(sigma_non), sigma_floor)

        sigma_sub = np.sqrt(np.sum(posterior_sub * (x - mu_sub) ** 2) / (N_sub + 1e-12))
        sigma_sub = max(float(sigma_sub), sigma_floor)

        # print(
        #     f"prior_non={prior_non:.3f}, prior_sub={prior_sub:.3f}, "
        #     f"sigma_non={sigma_non:.4f}, sigma_sub={sigma_sub:.4f}"
        # )


    # final E-step with final parameters
    p_non = prior_non * gaussian_pdf(x, mu_non, sigma_non)
    p_sub = prior_sub * gaussian_pdf(x, mu_sub, sigma_sub)

    posterior_non = p_non / (p_non + p_sub + 1e-12)
    posterior_sub = p_sub / (p_non + p_sub + 1e-12)

    return {
        "prior_non": prior_non,
        "prior_sub": prior_sub,
        "mu_non": mu_non,
        "mu_sub": mu_sub,
        "sigma_non": sigma_non,
        "sigma_sub": sigma_sub,
        "posterior_non": posterior_non,
        "posterior_sub": posterior_sub,
    }

In [5]:
def smooth_posterior(posterior, win_len):
    """
    Simple moving average smoothing.
    win_len should usually be odd: 3, 5, 7...
    """
    posterior = np.asarray(posterior, dtype=float)

    if win_len <= 1:
        return posterior.copy()

    kernel = np.ones(win_len, dtype=float) / win_len
    posterior_smooth = np.convolve(posterior, kernel, mode="same")
    return posterior_smooth

def find_runs(labels):
    labels = np.asarray(labels, dtype=bool)
    runs = []

    if len(labels) == 0:
        return runs

    start = 0
    current = labels[0]

    for i in range(1, len(labels)):
        if labels[i] != current:
            runs.append((current, start, i - 1))
            start = i
            current = labels[i]

    runs.append((current, start, len(labels) - 1))
    return runs

def fill_short_gaps(labels, min_gap_frames):
    """
    If a zero-run between two positive runs is short enough,
    convert it to ones.
    """
    labels = np.asarray(labels, dtype=bool).copy()
    runs = find_runs(labels)

    for value, start, end in runs:
        run_len = end - start + 1

        if value == False and run_len <= min_gap_frames:
            left_is_sub = start > 0 and labels[start - 1]
            right_is_sub = end < len(labels) - 1 and labels[end + 1]

            if left_is_sub and right_is_sub:
                labels[start:end + 1] = True

    return labels

def remove_short_subharmonic_runs(labels, min_len_frames):
    """
    Remove positive runs that are shorter than min_len_frames.
    """
    labels = np.asarray(labels, dtype=bool).copy()
    runs = find_runs(labels)

    for value, start, end in runs:
        run_len = end - start + 1

        if value == True and run_len < min_len_frames:
            labels[start:end + 1] = False

    return labels

def labels_to_intervals(times, labels):
    intervals = []
    in_interval = False
    start_idx = None

    for i, lab in enumerate(labels):
        # Start of subharmonic interval reached
        if lab == True and in_interval == False:
            in_interval = True
            start_idx = i

        # End of subharmonic interval reached
        elif lab == False and in_interval == True:
            end_idx = i - 1
            intervals.append((times[start_idx], times[end_idx]))
            in_interval = False

    # Subharmonic till the end
    if in_interval:
        intervals.append((times[start_idx], times[len(labels) - 1]))

    return intervals

def predAndPostProcessFrames(posteriors, decision_thresh, time_step, min_gap_frames, min_len_frames):
    smooth_span_sec = 0.05
    smooth_win_len = int(round(smooth_span_sec / time_step))
    if smooth_win_len % 2 == 0:
        smooth_win_len += 1
    smooth_win_len = max(smooth_win_len, 1)
    # print(f"Smooth win len for 50ms: {smooth_win_len} frames")

    # print(em["posterior_sub"].shape)
    posterior_smoothed = smooth_posterior(posteriors, smooth_win_len)
    # print(posterior_smoothed.shape)
    # print(time_frames.shape)
    frame_preds = (posterior_smoothed >= decision_thresh)
    # print(labels_sub.sum())

    frame_preds = fill_short_gaps(frame_preds, min_gap_frames)
    # print(f"After 1st gap filling: {np.sum(labels_sub)}")
    frame_preds = remove_short_subharmonic_runs(frame_preds, min_len_frames)
    # print(f"After removing short runs: {np.sum(labels_sub)}")
    frame_preds = fill_short_gaps(frame_preds, min_gap_frames)
    # print(f"After 2nd gap filling: {np.sum(labels_sub)}")

    return frame_preds

In [6]:
def detectSubharmonics(data, sample_length, tmin_cutoff, win_len, fref_mode, min_gap_dur, min_subharm_dur):

    # print(interval_labels)
    times = data[:, 0]
    pitch = data[:, 1]
    pitch = interpolate_pitch(pitch)

    if len(times) != len(pitch):
        raise IndexError("Time indeces don't match pitch values!")

    
    time_step = np.round(np.median(np.diff(times)), 4)

    # win_len = 0.25 # 250 ms
    n_side = int(round((win_len / 2) / time_step))
    # print(n_side)
    # print(np.searchsorted(times, 1, side='left'))
    # print(len(times) - n_side -1)
    # print(np.searchsorted(times, sample_length-1, side='right') - 1)
    first_frame_idx = max(n_side, np.searchsorted(times, tmin_cutoff, side='left')) # frame_idx = np.searchsorted(times, times[0] + win_len/2)
    last_frame_idx = min(len(times) - n_side -1, np.searchsorted(times, sample_length-tmin_cutoff, side='right') - 1)
    # win_size = 2*n_side + 1

    x_frames = getXtFrames(pitch, first_frame_idx, last_frame_idx, n_side, fref_mode)
    time_frames = times[first_frame_idx:last_frame_idx+1]

    prior_non_init, prior_sub_init = 0.9, 0.1
    em = em_two_fixed_means(x_frames, prior_non_init, prior_sub_init)   

    # min_subharm_dur = 0.05
    # max_gap_dur = 0.5

    min_len_frames = int(np.ceil(min_subharm_dur / time_step))
    min_gap_frames = int(np.floor(min_gap_dur / time_step))
    # print(f"min_len_frames = {min_len_frames}")
    # print(f"max_gap_frames = {max_gap_frames}")

    decision_thresh = 0.5
    # labels_sub = (em["posterior_sub"] >= decision_thresh)
    # print(labels_sub.sum())

    frame_preds = predAndPostProcessFrames(em["posterior_sub"], decision_thresh, time_step, min_gap_frames, min_len_frames)

    interval_preds = labels_to_intervals(time_frames, frame_preds)
    # print(f"Total number of subharmonic intervals: {len(intervals)}")

    return interval_preds

In [7]:
recording_lengths = {}  

groups = ["HC", "RBD", "PN", "MSA"]
for group in groups:
    folder_path = Path(group)/"prodlouzena_fonace"
    if folder_path.exists():
        for file_path in folder_path.iterdir():
            if file_path.is_file() and file_path.suffix.lower() == ".wav":

                fs, data = wavfile.read(file_path)
                sample_name = file_path.stem

                rec_duration = len(data)/fs
                recording_lengths[sample_name] = rec_duration


# df_labels = pd.read_csv("subharm_times.csv")

# # fix missing sample names
# df_labels["sample"] = df_labels["sample"].ffill()

# for sample, group_df in df_labels.groupby("sample"):
#     # intervals = list(zip(group_df["tmin"], group_df["tmax"]))
#     rec_len = recording_lengths[sample]
#     intervals = [
#                 (tmin, tmax)
#                 for tmin, tmax in zip(group_df["tmin"], group_df["tmax"])
#                 if (tmin > 1) and (tmin < rec_len - 1)
#                 ]

#     group_name = group_df["group"].iloc[0]  # same for whole sample
#     subharm_group_counts[group_name] += len(intervals)

#     labels_intervals[sample] = intervals # "group": group_name,

# print("Total number of subharmonic labels per group:")
# for g, gc in subharm_group_counts.items():
#     print(f"{g}: {gc}")

In [8]:

# performance_metrics = {"HC": {"TP": 0, "FP": 0, "FN": 0, "accuracy": 0, "precision": 0, "recall": 0, "F1": 0},
#                        "RBD": {"TP": 0, "FP": 0, "FN": 0, "accuracy": 0, "precision": 0, "recall": 0, "F1": 0},
#                        "PN": {"TP": 0, "FP": 0, "FN": 0, "accuracy": 0, "precision": 0, "recall": 0, "F1": 0},
#                        "MSA": {"TP": 0, "FP": 0, "FN": 0, "accuracy": 0, "precision": 0, "recall": 0, "F1": 0},
#                        "overall": {"TP": 0, "FP": 0, "FN": 0, "accuracy": 0, "precision": 0, "recall": 0, "F1": 0},}

# FP_group_sample_counter = {"HC": 0, "RBD": 0, "PN": 0, "MSA": 0}
# top3_FP_per_group = {"HC": [(None, -np.inf), (None, -np.inf), (None, -np.inf)], 
#                      "RBD": [(None, -np.inf), (None, -np.inf), (None, -np.inf)],
#                      "PN": [(None, -np.inf), (None, -np.inf), (None, -np.inf)], 
#                      "MSA": [(None, -np.inf), (None, -np.inf), (None, -np.inf)]}

# FN_group_sample_counter = {"HC": 0, "RBD": 0, "PN": 0, "MSA": 0}
# top3_FN_per_group = {"HC": [(None, -np.inf), (None, -np.inf), (None, -np.inf)], 
#                      "RBD": [(None, -np.inf), (None, -np.inf), (None, -np.inf)],
#                      "PN": [(None, -np.inf), (None, -np.inf), (None, -np.inf)], 
#                      "MSA": [(None, -np.inf), (None, -np.inf), (None, -np.inf)]}

# folder_path = Path("praat_pitches/vowels")
# if folder_path.exists():
#     for subfolder in folder_path.iterdir():
#         if subfolder.is_dir():
            
#             group = subfolder.name
#             # TP_total = 0
#             # FP_total = 0
#             # FN_total = 0
#             for file_path in subfolder.iterdir():
                
#                 if file_path.is_file() and file_path.suffix == ".txt":
                    
                    
#                     sample_name = file_path.name[len("Pitch "):-4]
#                     sample_length = recording_lengths[sample_name]
#                     data = np.loadtxt(file_path, delimiter="\t")

#                     sample_labels = labels_intervals[sample_name] if sample_name in labels_intervals else []
#                     tmin_cutoff = 1.
#                     TP, FP, FN = detectSubharmonics(data, sample_labels, sample_length, tmin_cutoff)
#                     # if FP > 0:
#                     #     FP_group_sample_counter[group] += 1
                    
#                     # if FP > top3_FP_per_group[group][0][1]:
#                     #     top3_FP_per_group[group] = [(sample_name, FP), top3_FP_per_group[group][0], top3_FP_per_group[group][1]]
#                     # elif FP > top3_FP_per_group[group][1][1]:
#                     #     top3_FP_per_group[group] = [top3_FP_per_group[group][0], (sample_name, FP), top3_FP_per_group[group][1]]
#                     # elif FP > top3_FP_per_group[group][2][1]: 
#                     #     top3_FP_per_group[group][2] = (sample_name, FP)

#                     # if FN > 0:
#                     #     FN_group_sample_counter[group] += 1
                    
#                     # if FN > top3_FN_per_group[group][0][1]:
#                     #     top3_FN_per_group[group] = [(sample_name, FN), top3_FN_per_group[group][0], top3_FN_per_group[group][1]]
#                     # elif FN > top3_FN_per_group[group][1][1]:
#                     #     top3_FN_per_group[group] = [top3_FN_per_group[group][0], (sample_name, FN), top3_FN_per_group[group][1]]
#                     # elif FN > top3_FN_per_group[group][2][1]: 
#                     #     top3_FN_per_group[group][2] = (sample_name, FN)

#                     performance_metrics[group]["TP"] += TP
#                     performance_metrics[group]["FP"] += FP
#                     performance_metrics[group]["FN"] += FN

#             accuracy = getAccuracy(performance_metrics[group]["TP"], performance_metrics[group]["FP"], performance_metrics[group]["FN"])
#             performance_metrics[group]["accuracy"] = accuracy
#             precision = getPrecision(performance_metrics[group]["TP"], performance_metrics[group]["FP"])
#             performance_metrics[group]["precision"] = precision
#             recall = getRecall(performance_metrics[group]["TP"], performance_metrics[group]["FN"])
#             performance_metrics[group]["recall"] = recall
#             f1 = getF1(precision, recall)
#             performance_metrics[group]["f1"] = f1

# for g in performance_metrics:
#     if g != "overall":
#         performance_metrics["overall"]["TP"] += performance_metrics[g]["TP"]
#         performance_metrics["overall"]["FP"] += performance_metrics[g]["FP"]
#         performance_metrics["overall"]["FN"] += performance_metrics[g]["FN"]

# performance_metrics["overall"]["accuracy"] = getAccuracy(performance_metrics["overall"]["TP"], performance_metrics["overall"]["FP"], performance_metrics["overall"]["FN"])
# performance_metrics["overall"]["precision"] = getPrecision(performance_metrics["overall"]["TP"], performance_metrics["overall"]["FP"])
# performance_metrics["overall"]["recall"] = getRecall(performance_metrics["overall"]["TP"], performance_metrics["overall"]["FN"])
# performance_metrics["overall"]["f1"] = getF1(performance_metrics["overall"]["precision"], performance_metrics["overall"]["recall"])

# for g in performance_metrics:         
#     print(f"---------{g}---------")
#     print(f"TP for {g}:", performance_metrics[g]["TP"])
#     print(f"FP for {g}:", performance_metrics[g]["FP"])
#     print(f"FN for {g}:", performance_metrics[g]["FN"])
#     print(f"Accuracy for {g}: {performance_metrics[g]["accuracy"]:.3f}")
#     print(f"Precision for {g}: {performance_metrics[g]["precision"]:.3f}")
#     print(f"Recall for {g}: {performance_metrics[g]["recall"]:.3f}")
#     print(f"F1 for {g}: {performance_metrics[g]["f1"]:.3f}")

#     if g != "overall":
#         print(f"Samples with an FP: {FP_group_sample_counter[g]}")
#         for top in top3_FP_per_group[g]:
#             print(f"Sample {top[0]} with {top[1]} FPs")

#         print(f"Samples with an FN: {FN_group_sample_counter[g]}")
#         for top in top3_FN_per_group[g]:
#             print(f"Sample {top[0]} with {top[1]} FNs")
                    

In [9]:
# RUN ALGORITHM -> GET SUBHARMONIC INTERVALS [(a, b), (c, d), ...] -> SAVE TO DICT
groups = ["HC", "RBD", "PN", "MSA"]

t_cutoff = 1.0
win_len = 0.25
fref_mode= "upper_70_90_median"
min_gap=0.5
min_subharm_len=0.05

result_rows_phonations = []

# PHONATIONS
for group in groups:

    folder_path = Path(f"praat_pitches/vowels/{group}")

    if folder_path.exists():
        for file_path in folder_path.iterdir():
            if file_path.is_file() and file_path.suffix.lower() == ".txt":
                
                data = np.loadtxt(file_path, delimiter="\t")
                sample_name = file_path.stem.replace("Pitch ", "")

                tmin_cutoff = 1.
                sample_duration = recording_lengths[sample_name]

                subharms = detectSubharmonics(data, sample_duration, tmin_cutoff, win_len, fref_mode, min_gap, min_subharm_len)
                subharms = np.asarray(subharms, dtype=float)

                # print(sample_name)
                # print(sample_duration)
                # print(subharms)
                # for subharm in subharms:
                #     print(f"{subharm[0]:.3f} {subharm[1]:.3f}")

                # ABSOLUTE NUMBER OF SUBHARMONICS IN THE SIGNAL
                subharm_total_no = len(subharms)
                # print(f"TOTAL NO. OF SUBHARMONICS: {subharm_total_no}")

                # SUBHARMONICS DURATION : SIGNAL DURATION RATIO
                subharm_ratio, subharm_per_second, subharm_durations, density_slope, pct_subharm_num_second_half, dur_density_slope, pct_subharm_dur_second_half = subharmRatio(subharm_total_no, subharms, sample_duration)
                # print(f"SUBHARMONICS-SIGNAL RATIO: {subharm_ratio}")

                subharm_durations = np.asarray(subharm_durations, dtype=float)

                # FIRST SUBHARMONIC OCCURENCE
                first_subharm_occur = firstSubharmOccur(subharm_total_no, subharms, sample_duration)
                # print(f"SUBHARMONIC FIRST OCCURENCE: {first_subharm_occur} seconds")

                # SUBHARMONIC DURATION STATS
                avg_subharm_dur, median_subharm_dur, std_subharm_dur, coeffOfVar_subharm_dur, longest_subharm_dur = statsSubharmDur(subharm_total_no, subharm_durations)
                # print(f"AVERAGE SUBHARMONIC DURATION: {avg_subharm_dur} seconds")    

                # INTER-SUBHARMONIC INTERVALS      
                mean_intervals, median_intervals, std_intervals, COV_intervals = interSubharmIntervals(subharm_total_no, subharms)
                # print(mean_intervals, median_intervals, std_intervals, COV_intervals)

                result_rows_phonations.append(
                    (sample_name, group,
                    subharm_total_no, 
                    subharm_ratio, subharm_per_second,
                    density_slope, pct_subharm_num_second_half, dur_density_slope, pct_subharm_dur_second_half,
                    first_subharm_occur, 
                    avg_subharm_dur, median_subharm_dur, std_subharm_dur, coeffOfVar_subharm_dur, longest_subharm_dur,
                    mean_intervals, median_intervals, std_intervals, COV_intervals))
                

df_results_phonations = pd.DataFrame(
    result_rows_phonations,
    columns=["sample", "group", "subharm_num", "subharm_sig_ratio", "subharm_per_second",
    "density_num_slope", "pct_subharm_num_2nd_half", "density_dur_slope", "pct_subharm_dur_2nd_half",
    "first_occur", 
    "avg_dur", "median_dur", "std_dur", "CV_dur", "longest_dur",
    "mean_inter_intervals", "median_inter_intervals", "std_inter_intervals", "COV_inter_intervals"])

df_results_phonations.to_csv("phonation_features.csv", index=False)





# win_len = 0.25
# fref_mode = "upper_70_90_median"
# min_gap = 0.5
# min_subharm_len = 0.05

# runEpoch(win_len, fref_mode, min_gap, min_subharm_len)

In [10]:
# # Sample HC845i2 with 7 FPs
# # Sample HC592i1 with 5 FPs
# # Sample HC861a2 with 4 FPs
# # Samples with an FN: 13
# # Sample HC592i1 with 5 FNs
# # Sample HC853i2 with 5 FNs
# # Sample HC836i2 with 4 FNs

# # Sample RBD204i1 with 5 FPs
# # Sample RBD114a1 with 4 FPs
# # Sample RBD130a1 with 3 FPs
# # Samples with an FN: 5
# # Sample RBD223a1 with 2 FNs
# # Sample RBD114i1 with 1 FNs
# # Sample RBD134i2 with 1 FNs

# data = np.loadtxt("praat_pitches/vowels/HC/Pitch HC853i2.txt", delimiter="\t")

# times = data[:, 0]
# pitch = data[:, 1]
# pitch = interpolate_pitch(pitch)
# plt.figure()
# plt.plot(times, pitch)
# plt.show()

# if len(times) != len(pitch):
#     raise IndexError("Time indeces don't match pitch values!")

# time_step = np.round(np.median(np.diff(times)), 4)
# print(f"Time step: {time_step}")

# win_len = 0.25 # 250 ms
# n_side = int(round((win_len / 2) / time_step))
# first_frame_idx = n_side # frame_idx = np.searchsorted(times, times[0] + win_len/2)
# last_frame_idx = len(pitch) - n_side -1
# # win_size = 2*n_side + 1

# x_frames = []
# for frame_idx in range(first_frame_idx, last_frame_idx+1):
#     x_t = getFrameXt(pitch, frame_idx, n_side, fref_mode="upper_median")
#     if x_t is None:
#         print("NONE")
#     x_frames.append(x_t)

# time_frames = times[first_frame_idx:last_frame_idx+1]

# # plt.figure()
# # plt.plot(time_frames, x_frames)
# # plt.ylim((-1.1, 0.1))
# # plt.show()

# x_frames = np.array(x_frames)
# em = em_two_fixed_means(x_frames)

# # # print("Estimated parameters:")
# # # print(f"prior_non  = {em['prior_non']:.3f}")
# # # print(f"prior_sub  = {em['prior_sub']:.3f}")
# # # print(f"sigma_non = {em['sigma_non']:.3f}")
# # # print(f"sigma_sub = {em['sigma_sub']:.3f}")



# # Min subharm length: 0.049 seconds
# # Min inter-subharm interval: 0.066 seconds

# time_step = np.median(np.diff(time_frames))
# print(f"time_step = {time_step:.4f} s")

# min_subharm_dur = 0.05
# max_gap_dur = 0.5

# min_len_frames = int(np.ceil(min_subharm_dur / time_step))
# max_gap_frames = int(np.floor(max_gap_dur / time_step))
# print(f"min_len_frames = {min_len_frames}")
# print(f"max_gap_frames = {max_gap_frames}")

# decision_thresh = 0.5
# labels_sub = (em["posterior_sub"] >= decision_thresh)
# print(labels_sub.sum())

# smooth_span_sec = 0.05
# smooth_win_len = int(round(smooth_span_sec / time_step))
# if smooth_win_len % 2 == 0:
#     smooth_win_len += 1
# smooth_win_len = max(smooth_win_len, 1)
# print(f"Smooth win len for 50ms: {smooth_win_len} frames")

# print(em["posterior_sub"].shape)
# posterior_smoothed = smooth_posterior(em["posterior_sub"], win_len)
# print(posterior_smoothed.shape)
# print(time_frames.shape)
# labels_sub = (posterior_smoothed >= decision_thresh)
# print(labels_sub.sum())

# labels_sub = fill_short_gaps(labels_sub, max_gap_frames)
# print(f"After 1st gap filling: {np.sum(labels_sub)}")
# labels_sub = remove_short_subharmonic_runs(labels_sub, min_len_frames)
# print(f"After removing short runs: {np.sum(labels_sub)}")
# labels_sub = fill_short_gaps(labels_sub, max_gap_frames)
# print(f"After 2nd gap filling: {np.sum(labels_sub)}")

# intervals = labels_to_intervals(time_frames, labels_sub)
# print(f"Total number of subharmonic intervals: {len(intervals)}")

# gaps = [intervals[i+1][0] - intervals[i][1] for i in range(len(intervals)-1)]
# print("Inter-subharmonic gaps [s]: ")
# for g in gaps:
#     print(g)

# plt.figure(figsize=(12, 6))

# plt.subplot(3, 1, 1)
# plt.plot(time_frames, x_frames)
# plt.title("x_t")
# plt.ylim([-1.1, 0.1])
# plt.grid(True)

# plt.subplot(3, 1, 2)
# plt.plot(time_frames, em["posterior_sub"], label="raw posterior")
# plt.plot(time_frames, posterior_smoothed, label="smoothed posterior")
# plt.axhline(y=decision_thresh, linestyle='--', color='k', label='threshold')
# plt.title("Subharmonic posterior")
# plt.legend()
# plt.ylim([-0.1, 1.1])
# plt.grid(True)

# plt.subplot(3, 1, 3)
# plt.plot(time_frames, labels_sub.astype(int))
# plt.title("Final binary labels")
# plt.ylim([-0.1, 1.1])
# plt.grid(True)

# for s, e in intervals:
#     plt.subplot(3, 1, 1)
#     plt.axvline(x=s, linestyle='--', color='r')
#     plt.axvline(x=e, linestyle='--', color='b')

#     plt.subplot(3, 1, 2)
#     plt.axvline(x=s, linestyle='--', color='r')
#     plt.axvline(x=e, linestyle='--', color='b')

#     plt.subplot(3, 1, 3)
#     plt.axvline(x=s, linestyle='--', color='r')
#     plt.axvline(x=e, linestyle='--', color='b')

# plt.tight_layout()
# plt.show()
